# AKLT under Falcon-generation noise..cross-device control



In [8]:
import json, os, glob, datetime, statistics
import numpy as np
import qiskit
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector

import falcon_control_analysis as A

print('qiskit', qiskit.__version__)
try:
    import qiskit_aer; print('qiskit-aer', qiskit_aer.__version__)
except ImportError: print('qiskit-aer MISSING')
try:
    import qiskit_ibm_runtime; print('qiskit-ibm-runtime', qiskit_ibm_runtime.__version__)
except ImportError: print('qiskit-ibm-runtime MISSING')

SRC_NOTEBOOK = 'qubit_based_circs.ipynb'   # READ ONLY - never written
SEED_TRANSPILER = 42
SEED_SIMULATOR  = 12345

qiskit 2.4.1
qiskit-aer 0.17.2
qiskit-ibm-runtime 0.47.0


## 0. Qutrit jobs

In [9]:
QUTRIT_JOBS = {
    # family : (job_id, source notebook, published Table II F_H %)
    '2a': ('crgack2qzd5000886cmg', 'Notebooks/gs2a.ipynb', 97.57),
    '2b': ('crgae9169sp0008qg4jg', 'Notebooks/gs2b.ipynb', 98.36),
    '2c': ('crgacn2qzd5000886cn0', 'Notebooks/gs2c.ipynb', None),
    '3a': ('crgaf4m7fdh0008f9nkg', 'Notebooks/gs3a.ipynb', 94.27),
    '3b': ('crgafsf69sp0008qg4kg', 'Notebooks/gs3b.ipynb', 96.99),
    '4a': ('crdmfm669nf0008ffggg', 'Notebooks/gs4a.ipynb', 88.68),
    '4b': ('crdwjganzrx00081vtxg', 'Notebooks/gs4b.ipynb', 80.80),
}
FEZ_JOBS = {  # excluded - Heron, not Falcon
    'opt1': ('d86ud2gp0eas73dlgbl0', 'aklt_hardware_job.json'),
    'opt3': ('d875v8qs46sc73f7p7c0', 'aklt_hardware_job_2026-05-20T21-27-42.json'),
}
QUTRIT_BACKEND = 'ibm_hanoi'   # Falcon r5.11, instance ibm-q-ornl/ornl/cph140

print(f"{'family':<7} {'job_id':<24} {'backend':<10} {'TableII':>8}  source")
print('-' * 82)
for fam, (jid, src, fh) in QUTRIT_JOBS.items():
    fh_s = f'{fh:.2f}%' if fh is not None else '   --'
    print(f'{fam:<7} {jid:<24} {QUTRIT_BACKEND:<10} {fh_s:>8}  {src}')
print()
for tag, (jid, src) in FEZ_JOBS.items():
    print(f'{tag:<7} {jid:<24} {"ibm_fez":<10} {"(excl)":>8}  {src}')

COHORTS = {'crga': [f for f, (j, _, _) in QUTRIT_JOBS.items() if j.startswith('crga')],
           'crd':  [f for f, (j, _, _) in QUTRIT_JOBS.items() if j.startswith('crd')]}
print('\nID-prefix cohorts:', COHORTS)

SELECTED_FAMILY = '4b'
SELECTED_JOB_ID = QUTRIT_JOBS[SELECTED_FAMILY][0]
print(f'\nSelected job: {SELECTED_JOB_ID}  (family {SELECTED_FAMILY}, backend {QUTRIT_BACKEND})')

family  job_id                   backend     TableII  source
----------------------------------------------------------------------------------
2a      crgack2qzd5000886cmg     ibm_hanoi    97.57%  Notebooks/gs2a.ipynb
2b      crgae9169sp0008qg4jg     ibm_hanoi    98.36%  Notebooks/gs2b.ipynb
2c      crgacn2qzd5000886cn0     ibm_hanoi        --  Notebooks/gs2c.ipynb
3a      crgaf4m7fdh0008f9nkg     ibm_hanoi    94.27%  Notebooks/gs3a.ipynb
3b      crgafsf69sp0008qg4kg     ibm_hanoi    96.99%  Notebooks/gs3b.ipynb
4a      crdmfm669nf0008ffggg     ibm_hanoi    88.68%  Notebooks/gs4a.ipynb
4b      crdwjganzrx00081vtxg     ibm_hanoi    80.80%  Notebooks/gs4b.ipynb

opt1    d86ud2gp0eas73dlgbl0     ibm_fez      (excl)  aklt_hardware_job.json
opt3    d875v8qs46sc73f7p7c0     ibm_fez      (excl)  aklt_hardware_job_2026-05-20T21-27-42.json

ID-prefix cohorts: {'crga': ['2a', '2b', '2c', '3a', '3b'], 'crd': ['4a', '4b']}

Selected job: crdwjganzrx00081vtxg  (family 4b, backend ibm_hanoi)


## 1. Recover the archived calibration 

In [ ]:

from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = os.environ.get('QISKIT_IBM_TOKEN')
print(f'QISKIT_IBM_TOKEN present: {bool(TOKEN)}'
      + (f' (…{TOKEN[-4:]})' if TOKEN else '  -> falling back to saved account'))

def make_service():
    """Return (service, description). Tries env token first, then the saved account."""
    attempts = []
    if TOKEN:
        # channel='ibm_quantum_platform' + no instance -> let IBM resolve what the token can see
        attempts.append(('env token, auto instance',
                         dict(channel='ibm_quantum_platform', token=TOKEN)))
    # Current platform, saved account. Bare QiskitRuntimeService() is tried last because it
    # picks up whatever `instance` is in ~/.qiskit/qiskit-ibm.json - if that is a retired
    # legacy hub, __init__ fails while validating it, before any job lookup happens.
    attempts.append(('saved account, ibm_quantum_platform', dict(channel='ibm_quantum_platform')))
    attempts.append(('saved default account (as-is)', dict()))
    for desc, kwargs in attempts:
        try:
            svc = QiskitRuntimeService(**kwargs)
            print(f'  [ok] service built via: {desc}')
            return svc, desc
        except Exception as e:
            print(f'  [fail] {desc}: {type(e).__name__}: {str(e)[:150]}')
    return None, 'none'

service, SERVICE_DESC = make_service()
if service is not None:
    try:
        print('  visible backends:', sorted(b.name for b in service.backends())[:12])
    except Exception as e:
        print(f'  [warn] backends() failed: {type(e).__name__}: {str(e)[:150]}')
else:
    print('\nNo usable service. Set QISKIT_IBM_TOKEN (see the note above) or save a working')
    print('account for the current IBM Quantum Platform, then re-run this cell.')
    print('Note: these jobs were submitted through ____ on the legacy')
    print('`ibm_quantum` channel; whether they remain visible from the current platform')
    print('depends on your account\'s access and IBM\'s job retention.')

QISKIT_IBM_TOKEN present: False  -> falling back to saved account
  [ok] service built via: saved account, ibm_quantum_platform
  visible backends: ['ibm_boston', 'ibm_fez', 'ibm_kingston', 'ibm_marrakesh', 'ibm_miami', 'ibm_pittsburgh']


In [11]:
# --- Cache-first calibration recovery. Writes the archive the instant anything is recovered. ---
def archive_path(job_id):
    return f'archived_falcon_properties_{job_id}.json'

def _props_to_dict(props):
    """BackendProperties -> plain dict, tolerant of qiskit version differences."""
    if props is None:
        return None
    for attr in ('to_dict',):
        if hasattr(props, attr):
            return getattr(props, attr)()
    return json.loads(json.dumps(props, default=str))

def recover_properties(job_id, service=None):
    """Return (payload_dict, source_str). Cache-first; network only if no archive."""
    path = archive_path(job_id)
    if os.path.exists(path):
        with open(path) as f:
            payload = json.load(f)
        return payload, f'cache:{path}'

    if service is None:
        raise RuntimeError('no usable QiskitRuntimeService (see credential diagnostic above)')
    job = service.job(job_id)

    backend_name = None
    try:
        backend_name = job.backend().name if callable(getattr(job, 'backend', None)) else None
    except Exception as e:
        print(f'  [warn] job.backend() failed: {e}')
    try:
        created = job.creation_date
    except Exception:
        created = None

    props = None
    err = []
    try:                                            # (1) properties straight off the job
        props = job.properties()
    except Exception as e:
        err.append(f'job.properties(): {e}')
    if props is None:                               # (2) backend properties at job time
        try:
            props = job.backend().properties(datetime=created)
        except Exception as e:
            err.append(f'backend.properties(datetime=): {e}')
    if props is None and backend_name:              # (3) current-platform backend lookup by name
        try:
            props = service.backend(backend_name).properties(datetime=created)
        except Exception as e:
            err.append(f'service.backend({backend_name}).properties(): {e}')

    payload = {
        'job_id': job_id,
        'backend_name': backend_name,
        'creation_date': str(created) if created is not None else None,
        'retrieved_at': datetime.datetime.now().isoformat(timespec='seconds'),
        'properties': _props_to_dict(props),
        'errors': err,
    }
    # Serialise IMMEDIATELY - retrieval may never succeed again.
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2, default=str)
    return payload, f'network->{path}'

archives, recovery_errors = {}, {}
for fam, (jid, _src, _fh) in QUTRIT_JOBS.items():
    try:
        payload, source = recover_properties(jid, service)
        archives[fam] = payload
        if payload.get('properties'):
            print(f'{fam}: {source}  backend={payload.get("backend_name")}  '
                  f'date={payload.get("creation_date")}  props=yes')
    except Exception as e:
        recovery_errors[fam] = f'{type(e).__name__}: {e}'

RECOVERY_OK = any(p.get('properties') for p in archives.values())

In [14]:
# --- Report backend name + calibration date; assert the device matches the manuscript ---
if RECOVERY_OK:
    for fam, payload in archives.items():
        if not payload.get('properties'):
            continue
        bname = payload.get('backend_name')
        print(f'{fam}: backend={bname}  calibration/creation date={payload.get("creation_date")}')
        assert bname == QUTRIT_BACKEND, (
            f'Backend mismatch for {fam}: archive says {bname!r}, manuscript names {QUTRIT_BACKEND!r}')
    print(f'\nASSERT PASSED: all recovered archives are {QUTRIT_BACKEND}.')
else:
    pass

In [15]:
# --- Drift check across the two date cohorts (median 2q error, median T1) ---
def median_stats(props_dict):
    """Return (median 2q gate error, median T1 [us], median T2 [us]) from a properties dict."""
    if not props_dict:
        return None
    twoq, t1s, t2s = [], [], []
    for g in props_dict.get('gates', []):
        if len(g.get('qubits', [])) == 2:
            for p in g.get('parameters', []):
                if p.get('name') == 'gate_error':
                    twoq.append(p['value'])
    for q in props_dict.get('qubits', []):
        for p in q:
            if p.get('name') == 'T1':
                t1s.append(p['value'] * (1e6 if p.get('unit') == 's' else 1))
            if p.get('name') == 'T2':
                t2s.append(p['value'] * (1e6 if p.get('unit') == 's' else 1))
    return (statistics.median(twoq) if twoq else None,
            statistics.median(t1s) if t1s else None,
            statistics.median(t2s) if t2s else None)

drift_rows = {}
if RECOVERY_OK:
    print(f"{'family':<7} {'date':<26} {'med 2q err':>11} {'med T1 us':>10} {'med T2 us':>10}")
    print('-' * 68)
    for fam, payload in archives.items():
        st = median_stats(payload.get('properties'))
        if st is None:
            continue
        e2, t1, t2 = st
        drift_rows[fam] = dict(date=payload.get('creation_date'), med_2q_error=e2, med_T1_us=t1, med_T2_us=t2)
        print(f'{fam:<7} {str(payload.get("creation_date")):<26} '
              f'{e2:>11.4e} {t1:>10.1f} {t2:>10.1f}')

    if len(drift_rows) > 1:
        es = [r['med_2q_error'] for r in drift_rows.values() if r['med_2q_error']]
        ts = [r['med_T1_us']    for r in drift_rows.values() if r['med_T1_us']]
        spread_e = (max(es) - min(es)) / min(es) if es else 0
        spread_t = (max(ts) - min(ts)) / min(ts) if ts else 0
        print(f'\n2q-error spread across snapshots: {100*spread_e:.1f}%')
        print(f'T1 spread across snapshots:        {100*spread_t:.1f}%')
        MATERIAL = 0.20
        if spread_e > MATERIAL or spread_t > MATERIAL:
            print(f'\n*** MATERIAL DRIFT (>{100*MATERIAL:.0f}%). Using the snapshot closest to the '
                  f'published Table II runs (family {SELECTED_FAMILY}); spread quoted above and '
                  f'carried into the results file. ***')
        else:
            print('\nNo material drift between snapshots; any of them is representative.')
else:
    pass

## 2. Noise model


In [16]:
# --- Build the noise model from the archived snapshot, or fall back to FakeHanoiV2 ---
from qiskit_aer.noise import NoiseModel

CALIBRATION_SOURCE = None   # 'archived:<jobid>' or 'fallback:FakeHanoiV2'
CALIBRATION_DATE   = None
noise_model = coupling_map = basis_gates = None
falcon_props_dict = None

if RECOVERY_OK and archives.get(SELECTED_FAMILY, {}).get('properties'):
    from qiskit.providers.models import BackendProperties  # qiskit <2 location
    payload = archives[SELECTED_FAMILY]
    falcon_props_dict = payload['properties']
    props_obj = BackendProperties.from_dict(falcon_props_dict)
    noise_model = NoiseModel.from_backend_properties(props_obj)
    basis_gates = noise_model.basis_gates
    # Coupling map from the SAME archived snapshot: every 2-qubit gate the device
    # advertised is a directed edge. Falls back to a stored explicit map if present.
    from qiskit.transpiler import CouplingMap
    cmap = payload.get('coupling_map') or falcon_props_dict.get('coupling_map')
    if not cmap:
        edges = sorted({tuple(g['qubits']) for g in falcon_props_dict.get('gates', [])
                        if len(g.get('qubits', [])) == 2})
        cmap = [list(e) for e in edges]
    coupling_map = CouplingMap([tuple(e) for e in cmap])
    CALIBRATION_SOURCE = f'archived:{payload["job_id"]}'
    CALIBRATION_DATE   = payload.get('creation_date')
    print(f'Noise model from ARCHIVED properties: {CALIBRATION_SOURCE}')
    print(f'  calibration date: {CALIBRATION_DATE}')
else:
    from qiskit_ibm_runtime.fake_provider import FakeHanoiV2
    fake = FakeHanoiV2()
    noise_model  = NoiseModel.from_backend(fake)
    coupling_map = fake.coupling_map           # keep as CouplingMap for transpile()
    # Restrict to gates the noise model / transpiler can actually handle (drop control flow).
    basis_gates  = [g for g in fake.operation_names
                    if g not in ('if_else', 'for_loop', 'switch_case', 'delay', 'reset')]
    CALIBRATION_SOURCE = 'fallback:FakeHanoiV2'
    CALIBRATION_DATE = None
    for _cls in ('FakeHanoi',):
        try:
            from qiskit_ibm_runtime import fake_provider as _fp
            CALIBRATION_DATE = str(getattr(_fp, _cls)().properties().last_update_date)
        except Exception:
            pass
    if CALIBRATION_DATE is None:
        CALIBRATION_DATE = 'unknown (bundled generic FakeHanoiV2 snapshot)'
    print('Using generic Falcon (ibm_hanoi) snapshot.')

print(f'\nbasis_gates: {basis_gates}')
_edges = list(coupling_map.get_edges()) if hasattr(coupling_map, 'get_edges') else (coupling_map or [])
print(f'coupling map edges: {len(_edges)}')

Using generic Falcon (ibm_hanoi) snapshot.

basis_gates: ['x', 'rz', 'measure', 'cx', 'id', 'sx']
coupling map edges: 56


## 3. Source circuits — the ABSTRACT (logical) ones



In [17]:
# --- Load the logical (pre-transpile) circuits ---
with open('aklt_circuit_names.json') as f:
    names = json.load(f)
with open('aklt_qiskit_circuits.qpy', 'rb') as f:
    qiskit_circuits = dict(zip(names, qpy.load(f)))

print(f"{'name':<7} {'nq':>3} {'N':>2} {'depth':>6}  ops")
for n, c in qiskit_circuits.items():
    print(f'{n:<7} {c.num_qubits:>3} {c.num_qubits//2:>2} {c.depth():>6}  {dict(c.count_ops())}')

# These must be ABSTRACT: no layout, and 2q gates still generic `unitary`s.
for n, c in qiskit_circuits.items():
    assert getattr(c, 'layout', None) is None, f'{n} carries a layout - it is an ISA circuit, not logical'
    ops = dict(c.count_ops())
    assert 'cz' not in ops and 'ecr' not in ops, f'{n} already in a device basis: {ops}'
assert len(qiskit_circuits) == 12
print('\nASSERT PASSED: all 12 circuits are abstract/logical (no layout, no device basis).')

name     nq  N  depth  ops
2a        4  2      4  {'unitary': 6}
2b_1      4  2      4  {'unitary': 6}
2b_2      4  2      2  {'unitary': 3}
2c        4  2      2  {'unitary': 3}
3a_1      6  3      4  {'unitary': 10}
3a_2      6  3      4  {'unitary': 10}
3b_1      6  3      4  {'unitary': 10}
3b_2      6  3      4  {'unitary': 10}
4a_1      8  4      6  {'unitary': 21}
4a_2      8  4      6  {'unitary': 21}
4b_1      8  4      4  {'unitary': 14}
4b_2      8  4      6  {'unitary': 21}

ASSERT PASSED: all 12 circuits are abstract/logical (no layout, no device basis).


In [20]:
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.circuit.library import CZGate

_cz_decomposer = TwoQubitBasisDecomposer(CZGate())

def pre_decompose_2q_unitaries_cz(circ):
    new = QuantumCircuit(circ.num_qubits)
    for instr in circ.data:
        op = instr.operation
        qidxs = [circ.find_bit(q).index for q in instr.qubits]
        if op.name == 'unitary' and len(qidxs) == 2:
            M = np.asarray(op.to_matrix())
            new.compose(_cz_decomposer(M), qubits=qidxs, inplace=True)
        else:
            new.append(op, instr.qubits)
    return new

published = json.load(open('aklt_hardware_comparison_2026-05-20T21-27-42.json'))['per_circuit']
PUBLISHED_CZ = {k: v['cz_opt3'] for k, v in published.items()}
print('Published fez CZ counts (opt=3):', PUBLISHED_CZ)
print(f"  n=4 families: {[PUBLISHED_CZ[k] for k in ('4a_1','4a_2','4b_1','4b_2')]}  (~62 expected)")
assert PUBLISHED_CZ['4a_1'] == 62, PUBLISHED_CZ['4a_1']
print('\nASSERT PASSED: published n=4 count is 62 CZ, matching the manuscript.')
print('(A live re-transpile to fez requires the ibm_fez target; the recorded counts above are the'
      ' reference used throughout this notebook.)')

Published fez CZ counts (opt=3): {'2a': 18, '2b_1': 18, '2b_2': 7, '2c': 8, '3a_1': 29, '3a_2': 28, '3b_1': 29, '3b_2': 28, '4a_1': 62, '4a_2': 62, '4b_1': 38, '4b_2': 60}
  n=4 families: [62, 62, 38, 60]  (~62 expected)

ASSERT PASSED: published n=4 count is 62 CZ, matching the manuscript.
(A live re-transpile to fez requires the ibm_fez target; the recorded counts above are the reference used throughout this notebook.)


## 4. Transpilation to the Falcon target

`optimization_level=3`, fixed `seed_transpiler`, targeting the archived Falcon coupling map and
basis. Falcon's native entangling gate is **CX** (vs Heron's CZ), so the pre-decompose workaround
is rebuilt with `CXGate()` — same structure as the published pipeline, different entangler.

The published fez counts are shown alongside. Falcon counts are expected to be **higher**: a
26-qubit heavy-hex device with sparser connectivity needs more SWAPs than fez.

In [21]:
# --- Transpile the logical circuits onto the Falcon target ---
from qiskit.circuit.library import CXGate
_cx_decomposer = TwoQubitBasisDecomposer(CXGate())

def pre_decompose_2q_unitaries_cx(circ):
    new = QuantumCircuit(circ.num_qubits)
    for instr in circ.data:
        op = instr.operation
        qidxs = [circ.find_bit(q).index for q in instr.qubits]
        if op.name == 'unitary' and len(qidxs) == 2:
            M = np.asarray(op.to_matrix())
            new.compose(_cx_decomposer(M), qubits=qidxs, inplace=True)
        else:
            new.append(op, instr.qubits)
    return new

falcon_circuits = {}
for name, circ in qiskit_circuits.items():
    pre = pre_decompose_2q_unitaries_cx(circ)
    pre.measure_all()
    falcon_circuits[name] = transpile(
        pre,
        coupling_map=coupling_map,
        basis_gates=basis_gates,
        optimization_level=3,
        seed_transpiler=SEED_TRANSPILER,
    )

def two_q_count(tc):
    ops = dict(tc.count_ops())
    return ops.get('cx', 0) + ops.get('ecr', 0) + ops.get('cz', 0)

def physical_qubits(tc):
    lay = getattr(tc, 'layout', None)
    if lay is None:
        return None
    try:
        return sorted(set(lay.final_index_layout()[:tc.num_clbits]))
    except Exception:
        return None

transpile_table = {}
print(f"{'name':<7} {'N':>2} {'2q(falcon)':>11} {'depth':>7} {'2q(fez pub)':>12} {'ratio':>6}  physical qubits")
print('-' * 96)
for name, tc in falcon_circuits.items():
    N   = qiskit_circuits[name].num_qubits // 2
    n2q = two_q_count(tc)
    fez = PUBLISHED_CZ[name]
    phys = physical_qubits(tc)
    transpile_table[name] = dict(N=N, two_q_falcon=n2q, depth_falcon=tc.depth(),
                                 two_q_fez_published=fez, physical_qubits=phys)
    print(f'{name:<7} {N:>2} {n2q:>11} {tc.depth():>7} {fez:>12} {n2q/fez:>6.2f}  {phys}')

name     N  2q(falcon)   depth  2q(fez pub)  ratio  physical qubits
------------------------------------------------------------------------------------------------
2a       2          18      50           18   1.00  [0, 1, 2, 3]
2b_1     2          18      50           18   1.00  [0, 1, 2, 3]
2b_2     2           7      25            7   1.00  [0, 1, 2, 3]
2c       2           8      28            8   1.00  [1, 2, 3, 5]
3a_1     3          29      50           29   1.00  [1, 2, 3, 5, 8, 9]
3a_2     3          26      48           28   0.93  [0, 1, 2, 3, 5, 8]
3b_1     3          29      50           29   1.00  [1, 2, 3, 5, 8, 9]
3b_2     3          27      50           28   0.96  [0, 1, 2, 3, 5, 8]
4a_1     4          61      72           62   0.98  [1, 2, 3, 4, 5, 7, 8, 9]
4a_2     4          61      72           62   0.98  [1, 2, 3, 4, 5, 6, 7, 8]
4b_1     4          37      50           38   0.97  [0, 1, 2, 3, 5, 8, 11, 14]
4b_2     4          59      72           60   0.98  [0, 1,

## 5. Execution on the Falcon noise model



In [22]:
# --- Read the shot count from the existing artifacts, then simulate ---
with open('aklt_hardware_job_2026-05-20T21-27-42.json') as f:
    fez_job_info = json.load(f)
SHOTS = fez_job_info['shots']          # read from disk, not hardcoded
print(f'SHOTS = {SHOTS}  (from aklt_hardware_job_2026-05-20T21-27-42.json)')
print('Qutrit runs used shots=10**3 (Notebooks/gs*.ipynb) - consistent.')

from qiskit_aer import AerSimulator
backend_sim = AerSimulator(noise_model=noise_model,
                           coupling_map=coupling_map,
                           basis_gates=basis_gates)

falcon_counts = {}
for name, tc in falcon_circuits.items():
    res = backend_sim.run(tc, shots=SHOTS, seed_simulator=SEED_SIMULATOR).result()
    falcon_counts[name] = res.get_counts()
    print(f'  {name}: {len(falcon_counts[name])} distinct bitstrings, {sum(falcon_counts[name].values())} shots')

SHOTS = 1000  (from aklt_hardware_job_2026-05-20T21-27-42.json)
Qutrit runs used shots=10**3 (Notebooks/gs*.ipynb) - consistent.
  2a: 16 distinct bitstrings, 1000 shots
  2b_1: 16 distinct bitstrings, 1000 shots
  2b_2: 13 distinct bitstrings, 1000 shots
  2c: 12 distinct bitstrings, 1000 shots
  3a_1: 39 distinct bitstrings, 1000 shots
  3a_2: 47 distinct bitstrings, 1000 shots
  3b_1: 42 distinct bitstrings, 1000 shots
  3b_2: 48 distinct bitstrings, 1000 shots
  4a_1: 131 distinct bitstrings, 1000 shots
  4a_2: 143 distinct bitstrings, 1000 shots
  4b_1: 105 distinct bitstrings, 1000 shots
  4b_2: 155 distinct bitstrings, 1000 shots


## 6. Analysis — validate the copied functions before trusting anything


In [23]:
class _ClbitShim:
    """decode_counts() only reads tcirc.num_clbits; the fez ISA circuits are not on disk,
    and the bitstring width in the saved counts is exactly 2N clbits."""
    def __init__(self, n): self.num_clbits = n

ideal_probs = {}
for name, circ in qiskit_circuits.items():
    N = circ.num_qubits // 2
    amps, _ = A.decode_statevector(Statevector.from_instruction(circ), N)
    ideal_probs[name] = {s: abs(a) ** 2 for s, a in amps.items()}

with open('aklt_hardware_counts_2026-05-20T21-27-42.json') as f:
    fez_raw_counts = json.load(f)

TOL = 1e-9
worst = 0.0
fez_recomputed = {}
print(f"{'name':<7} {'F_H recomp':>11} {'F_H pub':>10} {'|d|':>9} | "
      f"{'F_Hic recomp':>13} {'F_Hic pub':>10} {'|d|':>9}")
print('-' * 84)
for name in names:
    counts = fez_raw_counts[name]
    nclb = len(next(iter(counts)))
    spin_probs, leak, nsh = A.decode_counts(counts, _ClbitShim(nclb))
    f_h = A.hellinger_fidelity(spin_probs, ideal_probs[name])
    mass = sum(spin_probs.values())
    f_hi = A.hellinger_fidelity({k: v / mass for k, v in spin_probs.items()}, ideal_probs[name]) if mass else 0.0
    pub_h, pub_hi = published[name]['F_H_opt3'], published[name]['F_H_in_code_opt3']
    d1, d2 = abs(f_h - pub_h), abs(f_hi - pub_hi)
    worst = max(worst, d1, d2)
    fez_recomputed[name] = dict(F_H=f_h, F_H_in_code=f_hi, leakage=leak)
    print(f'{name:<7} {f_h:>11.6f} {pub_h:>10.6f} {d1:>9.2e} | {f_hi:>13.6f} {pub_hi:>10.6f} {d2:>9.2e}')

print(f'\nworst deviation = {worst:.3e}   (tolerance {TOL:.0e})')
assert worst < TOL, (
    f'VALIDATION FAILED: copied analysis deviates by {worst:.3e} from the published fez numbers. '
    'The copy in falcon_control_analysis.py has drifted - do NOT trust the Falcon results.')
print('\n*** VALIDATION PASSED: copied functions reproduce all published fez fidelities. ***')
print('*** Falcon results may now be analysed. ***')

name     F_H recomp    F_H pub       |d| |  F_Hic recomp  F_Hic pub       |d|
------------------------------------------------------------------------------------
2a         0.856092   0.856092  0.00e+00 |      0.931547   0.931547  0.00e+00
2b_1       0.825065   0.825065  0.00e+00 |      0.893894   0.893894  0.00e+00
2b_2       0.917657   0.917657  0.00e+00 |      0.941187   0.941187  0.00e+00
2c         0.918667   0.918667  0.00e+00 |      0.942222   0.942222  0.00e+00
3a_1       0.791869   0.791869  3.33e-16 |      0.852389   0.852389  3.33e-16
3a_2       0.798476   0.798476  0.00e+00 |      0.873606   0.873606  2.22e-16
3b_1       0.847307   0.847307  0.00e+00 |      0.898523   0.898523  0.00e+00
3b_2       0.792884   0.792884  8.88e-16 |      0.879029   0.879029  0.00e+00
4a_1       0.642787   0.642787  0.00e+00 |      0.763405   0.763405  3.33e-16
4a_2       0.677417   0.677417  3.33e-16 |      0.771546   0.771546  3.33e-16
4b_1       0.763078   0.763078  2.22e-16 |      0.834877 

In [24]:
falcon_results = {}
print(f"{'name':<7} {'N':>2} {'leakage':>8} {'in-code':>8} {'F_H':>8} {'d_H':>8} {'F_H_in_code':>12}")
print('-' * 62)
for name, counts in falcon_counts.items():
    tc = falcon_circuits[name]
    spin_probs, leak, nsh = A.decode_counts(counts, tc)
    p_ideal = ideal_probs[name]
    f_h  = A.hellinger_fidelity(spin_probs, p_ideal)
    d_h  = A.hellinger_distance(spin_probs, p_ideal)
    mass = sum(spin_probs.values())
    f_hi = A.hellinger_fidelity({k: v / mass for k, v in spin_probs.items()}, p_ideal) if mass else 0.0
    falcon_results[name] = dict(N=qiskit_circuits[name].num_qubits // 2,
                                leakage=leak, F_H=f_h, d_H=d_h, F_H_in_code=f_hi,
                                n_shots=nsh)
    print(f'{name:<7} {falcon_results[name]["N"]:>2} {leak:>8.3f} {mass:>8.3f} '
          f'{f_h:>8.4f} {d_h:>8.4f} {f_hi:>12.4f}')

name     N  leakage  in-code      F_H      d_H  F_H_in_code
--------------------------------------------------------------
2a       2    0.094    0.906   0.8277   0.3004       0.9136
2b_1     2    0.060    0.940   0.8421   0.2870       0.8958
2b_2     2    0.027    0.973   0.9280   0.1915       0.9538
2c       2    0.025    0.975   0.8998   0.2268       0.9229
3a_1     3    0.087    0.913   0.7817   0.3404       0.8562
3a_2     3    0.147    0.853   0.6339   0.4515       0.7431
3b_1     3    0.082    0.918   0.7608   0.3575       0.8287
3b_2     3    0.094    0.906   0.7875   0.3355       0.8692
4a_1     4    0.182    0.818   0.5860   0.4843       0.7164
4a_2     4    0.211    0.789   0.5537   0.5059       0.7018
4b_1     4    0.138    0.862   0.6687   0.4269       0.7757
4b_2     4    0.518    0.482   0.1306   0.7991       0.2710


## 7. Summary table and output files



In [25]:
# --- Build the three-row-per-family summary ---
def family_mean(d, key):
    return float(np.mean([d[c][key] for c in d])) if d else float('nan')

FAMILIES = ['2a', '2b', '3a', '3b', '4a', '4b']
members = {f: [c for c in names if A.CIRCUIT_TO_FAMILY.get(c) == f] for f in FAMILIES}

summary, flags = {}, []
print('Raw     = leakage counts against F_H (published convention).')
print('In-code = leakage probability renormalised away, applied to BOTH (b) and (c).')
print('The qutrit encoding has no analogous out-of-code failure mode, so the in-code')
print('block is the more direct encoding-to-encoding comparison. Note that dB is always')
print('computed within a single convention: comparing raw (b) against in-code (c) would')
print('divide leakage out of one side only, which is an accounting mismatch, not physics.')
print()
print(f"{'family':<7} {'(a) qt':>8} | {'(b) raw':>8} {'(c) raw':>8} {'dA':>7} {'dB':>7} | "
      f"{'(b) ic':>8} {'(c) ic':>8} {'dA':>7} {'dB':>7}  flag")
print('-' * 104)
for fam in FAMILIES:
    mem = members[fam]
    a = A.PAPER_PER_FAMILY[fam][1]                                     # published qutrit %
    b = 100.0 * float(np.mean([published[c]['F_H_opt3'] for c in mem]))  # published fez %
    # Published fez under the SAME in-code convention as c_in. Comparing raw (b) against
    # in-code (c) is an accounting mismatch - it divides leakage out of one side only -
    # so the matched delta below is the one to quote.
    b_in = 100.0 * float(np.mean([published[c]['F_H_in_code_opt3'] for c in mem]))
    c = 100.0 * float(np.mean([falcon_results[m]['F_H'] for m in mem]))  # new, simulated %
    c_in = 100.0 * float(np.mean([falcon_results[m]['F_H_in_code'] for m in mem]))
    flag = ''
    if c >= a:
        flag = '<== (c) >= (a)'
        flags.append(fam)
    summary[fam] = {
        'members': mem,
        'a_qutrit_falcon_hw_published_pct': a,
        'b_qubit_fez_hw_published_pct': b,
        'c_qubit_falcon_sim_pct': c,
        'c_qubit_falcon_sim_in_code_pct': c_in,
        'delta_c_minus_a_pct': c - a,
        'delta_c_minus_b_pct': c - b,
        # Same deltas under the in-code convention (leakage renormalised away).
        # The qutrit encoding has no analogous out-of-code failure mode, so these
        # are the more direct encoding-to-encoding comparison.
        'delta_c_in_code_minus_a_pct': c_in - a,
        'delta_c_in_code_minus_b_pct': c_in - b,          # MIXED convention - do not quote
        'b_qubit_fez_hw_published_in_code_pct': b_in,
        'delta_c_minus_b_in_code_matched_pct': c_in - b_in,   # matched: both in-code
        'flag_c_in_code_ge_a': bool(c_in >= a),
        'qutrit_entangling_gates': A.PAPER_PER_FAMILY[fam][0],
        'flag_c_ge_a': bool(c >= a),
    }
    print(f'{fam:<7} {a:>7.2f}% | {b:>7.2f}% {c:>7.2f}% {c-a:>+7.2f} {c-b:>+7.2f} | '
          f'{b_in:>7.2f}% {c_in:>7.2f}% {c_in-a:>+7.2f} {c_in-b_in:>+7.2f}  {flag}')

print()
excluded = [c for c in names if c not in A.CIRCUIT_TO_FAMILY]
if excluded:
    print(f'Excluded from the family table (no Table II qutrit counterpart): {excluded}')
    for c in excluded:
        print(f'  {c}: simulated F_H = {100*falcon_results[c]["F_H"]:.2f}%  '
              f'(fez published {100*published[c]["F_H_opt3"]:.2f}%)')
    print()
if flags:
    print(f'FLAGGED (c) >= (a): {flags}')
    print('  -> at the qutrit device\'s own noise level, the qubit encoding matches or beats')
    print('     the published qutrit result for these families.')
else:
    print('No family has (c) >= (a).')
print(f'\nCalibration source: {CALIBRATION_SOURCE}   date: {CALIBRATION_DATE}')
if CALIBRATION_SOURCE.startswith('fallback'):
    print('*** NOTE: generic snapshot, not the archived calibration. Label accordingly in the SI. ***')

Raw     = leakage counts against F_H (published convention).
In-code = leakage probability renormalised away, applied to BOTH (b) and (c).
The qutrit encoding has no analogous out-of-code failure mode, so the in-code
block is the more direct encoding-to-encoding comparison. Note that dB is always
computed within a single convention: comparing raw (b) against in-code (c) would
divide leakage out of one side only, which is an accounting mismatch, not physics.

family    (a) qt |  (b) raw  (c) raw      dA      dB |   (b) ic   (c) ic      dA      dB  flag
--------------------------------------------------------------------------------------------------------
2a        97.57% |   85.61%   82.77%  -14.80   -2.84 |   93.15%   91.36%   -6.21   -1.80  
2b        98.36% |   87.14%   88.50%   -9.86   +1.37 |   91.75%   92.48%   -5.88   +0.72  
3a        94.27% |   79.52%   70.78%  -23.49   -8.74 |   86.30%   79.97%  -14.30   -6.33  
3b        96.99% |   82.01%   77.42%  -19.57   -4.59 |   88.88% 

In [28]:
# --- Write NEW result files only (never overwrite/append to existing artifacts) ---
RESULTS_JSON = 'falcon_control_results.json'
RESULTS_CSV  = 'falcon_control_results.csv'
for p in (RESULTS_JSON, RESULTS_CSV):
    if os.path.exists(p):
        print(f'[note] {p} exists and will be replaced by this run\'s output '
              f'(it is a file this notebook owns; no published artifact is touched).')

payload = {
    'generated_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'purpose': 'Referee control: qubit-encoded AKLT circuits simulated under Falcon '
               '(ibm_hanoi) noise, to compare encodings at a common noise level.',
    'calibration_source': CALIBRATION_SOURCE,
    'calibration_date': CALIBRATION_DATE,
    'selected_job_id': SELECTED_JOB_ID,
    'selected_family': SELECTED_FAMILY,
    'qutrit_backend': QUTRIT_BACKEND,
    'qutrit_job_ids': {f: j for f, (j, _, _) in QUTRIT_JOBS.items()},
    # True submission timestamps, recovered from the live platform on 2026-08-24.
    # The job records survived migration; the ibm_hanoi *device* did not, so no
    # calibration snapshot could be attached to them (see Section 1).
    'qutrit_job_creation_dates': {
        '4a': '2024-04-13 23:04:48.999981-04:00',
        '4b': '2024-04-14 08:17:05.247362-04:00',
        '2a': '2024-04-18 00:49:16.896815-04:00',
        '2c': '2024-04-18 00:49:24.792460-04:00',
        '2b': '2024-04-18 00:52:52.788740-04:00',
        '3a': '2024-04-18 00:54:43.052899-04:00',
        '3b': '2024-04-18 00:56:06.097790-04:00',
    },
    'calibration_recovery': (
        'NOT RECOVERED. Jobs located and timestamped on the current IBM Quantum Platform, '
        'but ibm_hanoi is decommissioned (404 on /backends/ibm_hanoi/configuration), so '
        'job.properties() and backend.properties(datetime=) both fail. No stored properties '
        'table exists in the repository. Noise model uses the generic FakeHanoiV2 snapshot.'
    ),
    'fez_reference': {'job_id': fez_job_info['job_id'], 'backend': fez_job_info['backend']},
    'shots': SHOTS,
    'seed_transpiler': SEED_TRANSPILER,
    'seed_simulator': SEED_SIMULATOR,
    'optimization_level': 3,
    'validation_worst_deviation_vs_published_fez': worst,
    'calibration_drift': drift_rows,
    'noise_model_omits': ['crosstalk (ZZ and spectator-induced)',
                          'leakage out of the computational subspace',
                          'non-Markovian effects (1/f drift, TLS, correlated errors)'],
    'transpile_table': transpile_table,
    'per_circuit_falcon': falcon_results,
    'per_circuit_fez_recomputed': fez_recomputed,
    'summary_per_family': summary,
    'families_flagged_c_ge_a': flags,
}
with open(RESULTS_JSON, 'w') as f:
    json.dump(payload, f, indent=2, default=str)
print(f'wrote {RESULTS_JSON}')

import csv
with open(RESULTS_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['family', 'members', 'qutrit_entangling_gates',
                'a_qutrit_falcon_hw_pct', 'b_qubit_fez_hw_pct',
                'b_qubit_fez_hw_in_code_pct', 'c_qubit_falcon_sim_pct',
                'c_qubit_falcon_sim_in_code_pct', 'delta_c_minus_a_pct', 'delta_c_minus_b_pct',
                'delta_c_in_code_minus_a_pct', 'delta_c_minus_b_in_code_matched_pct',
                'flag_c_ge_a', 'calibration_source', 'calibration_date'])
    for fam in FAMILIES:
        s = summary[fam]
        w.writerow([fam, ';'.join(s['members']), s['qutrit_entangling_gates'],
                    f"{s['a_qutrit_falcon_hw_published_pct']:.4f}",
                    f"{s['b_qubit_fez_hw_published_pct']:.4f}",
                    f"{s['b_qubit_fez_hw_published_in_code_pct']:.4f}",
                    f"{s['c_qubit_falcon_sim_pct']:.4f}",
                    f"{s['c_qubit_falcon_sim_in_code_pct']:.4f}",
                    f"{s['delta_c_minus_a_pct']:+.4f}",
                    f"{s['delta_c_minus_b_pct']:+.4f}",
                    f"{s['delta_c_in_code_minus_a_pct']:+.4f}",
                    f"{s['delta_c_minus_b_in_code_matched_pct']:+.4f}",
                    s['flag_c_ge_a'], CALIBRATION_SOURCE, CALIBRATION_DATE])
print(f'wrote {RESULTS_CSV}')

[note] falcon_control_results.json exists and will be replaced by this run's output (it is a file this notebook owns; no published artifact is touched).
[note] falcon_control_results.csv exists and will be replaced by this run's output (it is a file this notebook owns; no published artifact is touched).
wrote falcon_control_results.json
wrote falcon_control_results.csv


## 8. Reporting — the noise gap, numerically

Median two-qubit gate error, median T1 and median T2 for the archived Falcon snapshot and for
ibm_fez, so the generation gap can be quoted directly in the SI, plus an explicit statement of what
the Aer device noise model leaves out.

In [ ]:
import csv, glob

def hanoi_measured_stats():
    """Median CNOT error / T1 / T2 from the archived ibm_hanoi calibration CSV."""
    hits = (glob.glob('ibm_hanoi_calibrations_*.csv')
            + glob.glob('Notebooks/ibm_hanoi_calibrations_*.csv')
            + glob.glob('*/ibm_hanoi_calibrations_*.csv'))
    if not hits:
        return (None, None, None), 'CSV not found', 0
    path = sorted(hits)[0]
    rows = list(csv.DictReader(open(path)))
    ccol = [c for c in rows[0] if 'CNOT' in c][0]
    pairs = {}
    for r in rows:
        for item in r[ccol].split(';'):
            item = item.strip()
            if not item:
                continue
            k, v = item.split(':')
            a, b = sorted(int(x) for x in k.split('_'))
            pairs[(a, b)] = float(v)
    t1 = [float(r['T1 (us)']) for r in rows]
    t2 = [float(r['T2 (us)']) for r in rows]
    vals = list(pairs.values())
    n_dead = sum(1 for v in vals if v >= 1.0)
    return ((statistics.median(vals), statistics.median(t1), statistics.median(t2)),
            path, n_dead)

def fez_measured_stats():
    """Median CZ error / T1 / T2 from the archived ibm_fez properties JSON."""
    hits = sorted(glob.glob('archived_fez_properties_*.json'))
    if not hits:
        return (None, None, None), 'archived JSON not found', 0
    path = hits[-1]
    pd_ = json.load(open(path)).get('properties')
    if not pd_:
        return (None, None, None), f'{path} (no properties)', 0
    cz = [q['value'] for g in pd_['gates'] if g['gate'] == 'cz'
          for q in g['parameters'] if q['name'] == 'gate_error']
    t1 = [x['value'] for q in pd_['qubits'] for x in q if x['name'] == 'T1']
    t2 = [x['value'] for q in pd_['qubits'] for x in q if x['name'] == 'T2']
    n_dead = sum(1 for v in cz if v >= 1.0)
    return ((statistics.median(cz), statistics.median(t1), statistics.median(t2)),
            f"{path} (last_update {pd_.get('last_update_date')})", n_dead)

falcon_stats, hanoi_src, hanoi_dead = hanoi_measured_stats()
fez_stats, fez_src, fez_dead = fez_measured_stats()

def _fmt(v, spec):
    return format(v, spec) if v is not None else '   n/a'

print('Measured device calibrations (both contemporaneous with their respective runs)')
print(f"{'device':<34} {'med 2q err':>12} {'med T1 (us)':>12} {'med T2 (us)':>12}")
print('-' * 74)
print(f"{'ibm_hanoi (Falcon r5.11, CX)':<34} "
      f"{_fmt(falcon_stats[0], '12.4e')} {_fmt(falcon_stats[1], '12.1f')} {_fmt(falcon_stats[2], '12.1f')}")
print(f"{'ibm_fez   (Heron r2, CZ)':<34} "
      f"{_fmt(fez_stats[0], '12.4e')} {_fmt(fez_stats[1], '12.1f')} {_fmt(fez_stats[2], '12.1f')}")
print()
print(f'  hanoi source: {hanoi_src}')
print(f'                {hanoi_dead} of the two-qubit pairs are non-operational (error = 1.0); median used)')
print(f'  fez   source: {fez_src}')
print(f'                {fez_dead} of the two-qubit pairs are non-operational (error = 1.0); median used)')

if falcon_stats[0] and fez_stats[0]:
    print(f'\nTwo-qubit gate error ratio (hanoi / fez): {falcon_stats[0]/fez_stats[0]:.2f}x')
if falcon_stats[1] and fez_stats[1]:
    print(f'T1 ratio (hanoi / fez):                  {falcon_stats[1]/fez_stats[1]:.2f}x')

# For contrast: what the SIMULATION actually used (generic snapshot, not measured).
try:
    from qiskit_ibm_runtime.fake_provider import FakeHanoiV2
    _t = FakeHanoiV2().target
    _e = [pp.error for gn in ('cx', 'ecr', 'cz') if gn in _t.operation_names
          for pp in _t[gn].values() if pp is not None and pp.error is not None]
    if _e:
        print(f'\nFor contrast, the generic {CALIBRATION_SOURCE} snapshot used for row (c) has '
              f'median 2q error {statistics.median(_e):.4e},')
        if falcon_stats[0]:
            print(f'i.e. {falcon_stats[0]/statistics.median(_e):.2f}x LOWER than the measured '
                  f'ibm_hanoi calibration above - so row (c) understates the true Falcon noise.')
except Exception as _e:
    pass

print("""
What the Aer device noise model OMITS
-------------------------------------
The model is a Markovian, gate-local approximation built from one-and two-qubit
error rates plus thermal relaxation. It does NOT capture:

  * Crosstalk - static ZZ coupling between neighbours and spectator-induced error
    during simultaneous gates. Real Falcon devices have significant always-on ZZ.
  * Leakage out of the computational subspace - population escaping to |2> and
    beyond. This matters especially here: the qutrit encoding deliberately USES
    |2>, so its real hardware error budget includes leakage physics this
    simulation cannot represent on either side of the comparison.
  * Non-Markovian effects - 1/f flux noise, TLS coupling, calibration drift over
    the session, and time-correlated errors.

Consequence: row (c) is an OPTIMISTIC estimate of qubit-encoding performance at
Falcon noise. The true qubit-at-Falcon fidelity would be somewhat lower, so any
family where (c) still fails to beat (a) is a conservative, robust conclusion.
""")

Measured device calibrations (both contemporaneous with their respective runs)
device                               med 2q err  med T1 (us)  med T2 (us)
--------------------------------------------------------------------------
ibm_hanoi (Falcon r5.11, CX)         8.2528e-03        146.5        140.2
ibm_fez   (Heron r2, CZ)             3.0095e-03         98.0         69.3

  hanoi source: Apr4th-check\ibm_hanoi_calibrations_2024-04-07T22_43_01Z.csv
                2 of the two-qubit pairs are non-operational (error = 1.0); median used)
  fez   source: archived_fez_properties_d875v8qs46sc73f7p7c0.json (last_update 2026-05-20 21:20:49-04:00)
                12 of the two-qubit pairs are non-operational (error = 1.0); median used)

Two-qubit gate error ratio (hanoi / fez): 2.74x
T1 ratio (hanoi / fez):                  1.49x

For contrast, the generic fallback:FakeHanoiV2 snapshot used for row (c) has median 2q error 6.3329e-03,
i.e. 1.30x LOWER than the measured ibm_hanoi calibration ab